<a href="https://colab.research.google.com/github/vivekcodes-tech/CineSearch/blob/main/CineSearch_student3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import zipfile
import os

url = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
path = "/content/ml-32m.zip"

r = requests.get(url, timeout=300, verify=False)
r.raise_for_status()

with open(path, "wb") as f:
    f.write(r.content)

print("Downloaded file size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

with zipfile.ZipFile(path, "r") as z:
    z.extractall("/content/")

print("Dataset successfully downloaded and extracted!")

/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'files.grouplens.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Downloaded file size: 227.88 MB
Dataset successfully downloaded and extracted!


In [2]:
import pandas as pd
import os

movies_path = "/content/ml-32m/movies.csv"

movies = pd.read_csv(movies_path)

print("Total movies:", len(movies))
print("\nFirst 10 movies:")
display(movies.head(10))

print("\nColumn names:")
print(movies.columns.tolist())

Total movies: 87585

First 10 movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller



Column names:
['movieId', 'title', 'genres']


In [3]:
import pandas as pd

ratings_path = "/content/ml-32m/ratings.csv"

rating_parts = []

for chunk in pd.read_csv(ratings_path, chunksize=500000):
    summary = chunk.groupby("movieId").agg(
        rating_sum=("rating", "sum"),
        rating_count=("rating", "count")
    )
    rating_parts.append(summary)

all_ratings = pd.concat(rating_parts)

movie_stats = all_ratings.groupby("movieId").agg(
    rating_sum=("rating_sum", "sum"),
    rating_count=("rating_count", "sum")
).reset_index()

movie_stats["average_rating"] = (
    movie_stats["rating_sum"] / movie_stats["rating_count"]
)

movies = movies.merge(movie_stats, on="movieId", how="left")

movies["rating_count"] = movies["rating_count"].fillna(0)
movies["average_rating"] = movies["average_rating"].fillna(0)

print("Rating analysis completed!")
print("Movies with ratings:", (movies["rating_count"] > 0).sum())

display(
    movies[["title", "genres", "average_rating", "rating_count"]]
    .sort_values("rating_count", ascending=False)
    .head(10)
)

Rating analysis completed!
Movies with ratings: 84432


,title,genres,average_rating,rating_count
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0
475,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,3.698623,75233.0
522,Schindler's List (1993),Drama|War,4.236990,73849.0
4888,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,4.092134,73122.0


In [4]:
import numpy as np

# Normalize popularity using log to reduce the effect of extremely popular movies
movies["popularity_score"] = np.log1p(movies["rating_count"])

max_popularity = movies["popularity_score"].max()

if max_popularity > 0:
    movies["popularity_score"] = (
        movies["popularity_score"] / max_popularity
    )

# Normalize average rating to a score between 0 and 1
movies["rating_score"] = movies["average_rating"] / 5.0

# Composite ranking: combine popularity and average rating
movies["composite_score"] = (
    0.5 * movies["popularity_score"]
    + 0.5 * movies["rating_score"]
)

# Rank movies from highest to lowest composite score
movies["rank"] = (
    movies["composite_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

top_movies = movies.sort_values(
    "composite_score", ascending=False
).head(10)

print("Top 10 Movies by Composite Ranking")
display(
    top_movies[
        ["rank", "title", "genres", "average_rating",
         "rating_count", "popularity_score", "composite_score"]
    ]
)

Top 10 Movies by Composite Ranking


,rank,title,genres,average_rating,rating_count,popularity_score,composite_score
314,1,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,1.000000,0.940461
292,2,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.996109,0.917751
840,3,"Godfather, The (1972)",Crime|Drama,4.317030,66440.0,0.962074,0.912740
2480,4,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.991961,0.911624
2867,5,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.975227,0.910491
522,6,Schindler's List (1993),Drama|War,4.236990,73849.0,0.971234,0.909316
585,7,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.988687,0.909180
49,8,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070,67750.0,0.963766,0.908390
351,9,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.997755,0.904152
257,10,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.983428,0.901696


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Prepare movie title and genre text
titles = movies["title"].fillna("").astype(str)
genres = movies["genres"].fillna("").astype(str)

# TF-IDF for movie titles
title_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)
title_matrix = title_vectorizer.fit_transform(titles)

# TF-IDF for movie genres
genre_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b"
)
genre_matrix = genre_vectorizer.fit_transform(genres)

def search_movies(query, top_n=10):
    query = query.strip()

    if not query:
        print("Please enter a search query.")
        return

    title_query = title_vectorizer.transform([query])
    genre_query = genre_vectorizer.transform([query])

    title_scores = cosine_similarity(
        title_query, title_matrix
    ).flatten()

    genre_scores = cosine_similarity(
        genre_query, genre_matrix
    ).flatten()

    # Field-weighted relevance score
    search_scores = (
        0.7 * title_scores + 0.3 * genre_scores
    )

    results = movies.copy()
    results["search_score"] = search_scores

    results = results[results["search_score"] > 0]
    results = results.sort_values(
        "search_score", ascending=False
    ).head(top_n)

    print("Search results for:", query)

    if results.empty:
        print("No matching movies found.")
    else:
        display(
            results[
                ["title", "genres", "average_rating",
                 "rating_count", "search_score"]
            ]
        )

print("TF-IDF Search Ranking is ready!")
search_movies("Batman", 10)

TF-IDF Search Ranking is ready!
Search results for: Batman


,title,genres,average_rating,rating_count,search_score
46768,Batman & Bill (2017),Documentary,3.814815,27.0,0.397751
80120,The Batman (2022),Action|Crime|Drama,3.672605,3340.0,0.394296
584,Batman (1989),Action|Crime|Thriller,3.409920,49789.0,0.385933
8616,Batman (1966),Action|Adventure|Comedy,3.214971,1042.0,0.381484
17362,Batman (1943),Action|Adventure|Crime|Sci-Fi|Thriller,2.983333,30.0,0.375317
48161,Batman Beyond: The Movie (1999),Action|Animation|Children,3.500000,59.0,0.311715
72228,Heart of Batman (2018),Documentary,3.833333,3.0,0.296398
53894,Batman Ninja (2018),Action|Animation,2.937500,192.0,0.294648
151,Batman Forever (1995),Action|Adventure|Comedy|Crime,2.862687,35288.0,0.287450
1506,Batman & Robin (1997),Action|Adventure|Fantasy|Thriller,2.161005,11686.0,0.286160


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

movie_input = widgets.Text(
    placeholder="e.g. Batman, Avatar, Comedy",
    description="Search:",
    layout=widgets.Layout(width="70%")
)

search_button = widgets.Button(
    description="Search",
    button_style="primary"
)

results_output = widgets.Output()

def run_movie_search(button=None):
    with results_output:
        clear_output(wait=True)
        query = movie_input.value.strip()

        if query:
            search_movies(query, 10)
        else:
            print("Please enter a movie name or genre.")

search_button.on_click(run_movie_search)
movie_input.on_submit(run_movie_search)

display(widgets.VBox([
    widgets.HTML("<h2>CineSearch</h2><p>Search movies by title or genre</p>"),
    widgets.HBox([movie_input, search_button]),
    results_output
]))

In [7]:
# 1. Recommend popular movies
def recommend_popular(top_n=10):
    results = movies.sort_values(
        "composite_score", ascending=False
    ).head(top_n)

    print("Popular Movie Recommendations")
    display(results[
        ["title", "genres", "average_rating",
         "rating_count", "composite_score"]
    ])

# 2. Recommend movies by genre
def recommend_by_genre(genre, top_n=10):
    genre = genre.strip().lower()

    matching = movies[
        movies["genres"].str.lower().str.split("|").apply(
            lambda genres: genre in genres
        )
    ]

    results = matching.sort_values(
        "composite_score", ascending=False
    ).head(top_n)

    print("Recommendations for genre:", genre.title())

    if results.empty:
        print("No movies found for this genre.")
    else:
        display(results[
            ["title", "genres", "average_rating",
             "rating_count", "composite_score"]
        ])

print("Recommendation system is ready!")

recommend_popular(10)
recommend_by_genre("Comedy", 10)

Recommendation system is ready!
Popular Movie Recommendations


,title,genres,average_rating,rating_count,composite_score
314,"Shawshank Redemption, The (1994)",Crime|Drama,4.404614,102929.0,0.940461
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
840,"Godfather, The (1972)",Crime|Drama,4.317030,66440.0,0.912740
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.910491
522,Schindler's List (1993),Drama|War,4.236990,73849.0,0.909316
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.909180
49,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,4.265070,67750.0,0.908390
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696


Recommendations for genre: Comedy


,title,genres,average_rating,rating_count,composite_score
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
600,Fargo (1996),Comedy|Crime|Drama|Thriller,4.116533,58031.0,0.886828
1108,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,4.135633,46508.0,0.879149
1167,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957.0,0.877913
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997.0,0.872417
4868,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.082169,43459.0,0.870865
734,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.202680,32830.0,0.870766
2233,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.150374,29819.0,0.861369


In [8]:
recommend_by_genre("Action", 10)
recommend_by_genre("Comedy", 10)
recommend_by_genre("Horror", 10)
recommend_by_genre("Sci-Fi", 10)

Recommendations for genre: Action


,title,genres,average_rating,rating_count,composite_score
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
2867,Fight Club (1999),Action|Crime|Drama|Thriller,4.228780,77332.0,0.910491
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696
1166,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130352,72151.0,0.897644
12223,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX,4.173349,59334.0,0.893472
1168,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,4.111893,67408.0,0.892853
7029,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.094360,67449.0,0.891126
14939,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.157170,57931.0,0.890817
108,Braveheart (1995),Action|Drama|War,3.988630,69482.0,0.881839
1179,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991555,67496.0,0.880876


Recommendations for genre: Comedy


,title,genres,average_rating,rating_count,composite_score
292,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,4.196969,98409.0,0.917751
351,Forrest Gump (1994),Comedy|Drama|Romance|War,4.052744,100296.0,0.904152
600,Fargo (1996),Comedy|Crime|Drama|Thriller,4.116533,58031.0,0.886828
1108,Monty Python and the Holy Grail (1975),Adventure|Comedy|Fantasy,4.135633,46508.0,0.879149
1167,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.119109,46957.0,0.877913
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997.0,0.872417
4868,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,4.082169,43459.0,0.870865
734,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.202680,32830.0,0.870766
2233,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.150374,29819.0,0.861369


Recommendations for genre: Horror


,title,genres,average_rating,rating_count,composite_score
585,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,4.148367,90330.0,0.909180
2670,"Sixth Sense, The (1999)",Drama|Horror|Mystery,4.003210,57010.0,0.874727
1183,Alien (1979),Horror|Sci-Fi,4.068401,45211.0,0.871201
1225,"Shining, The (1980)",Horror,4.036439,38640.0,0.861201
1170,Aliens (1986),Action|Adventure|Horror|Sci-Fi,4.007259,38846.0,0.858513
1188,Psycho (1960),Crime|Horror,4.062512,27027.0,0.848324
8161,Shaun of the Dead (2004),Comedy|Horror,3.878019,26914.0,0.829693
1351,Jaws (1975),Action|Horror,3.785016,29265.0,0.824020
2197,"Thing, The (1982)",Action|Horror|Sci-Fi|Thriller,3.935967,14719.0,0.809345
6380,28 Days Later (2002),Action|Horror|Sci-Fi,3.772137,21390.0,0.809153


Recommendations for genre: Sci-Fi


,title,genres,average_rating,rating_count,composite_score
2480,"Matrix, The (1999)",Action|Sci-Fi|Thriller,4.156437,93808.0,0.911624
257,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,4.099824,85010.0,0.901696
1166,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.130352,72151.0,0.897644
14939,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,4.157170,57931.0,0.890817
1179,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,3.991555,67496.0,0.880876
581,Terminator 2: Judgment Day (1991),Action|Sci-Fi,3.962045,68383.0,0.878490
536,Blade Runner (1982),Action|Sci-Fi|Thriller,4.110005,46289.0,0.876382
1237,Back to the Future (1985),Adventure|Comedy|Sci-Fi,3.969149,61879.0,0.874871
1183,Alien (1979),Horror|Sci-Fi,4.068401,45211.0,0.871201
7237,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,4.065294,44430.0,0.870135


In [9]:
print("Available columns:")
print(movies.columns.tolist())

print("\nDataset shape:")
print(movies.shape)

print("\nAvailable genres:")
print(movies['genres'].head(10))

Available columns:
['movieId', 'title', 'genres', 'rating_sum', 'rating_count', 'average_rating', 'popularity_score', 'rating_score', 'composite_score', 'rank']

Dataset shape:
(87585, 10)

Available genres:
0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object


In [10]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Get all available genres
all_genres = set()

for genre_list in movies['genres'].dropna():
    for genre in genre_list.split('|'):
        if genre != '(no genres listed)':
            all_genres.add(genre)

# Create dropdown
genre_dropdown = widgets.Dropdown(
    options=sorted(all_genres),
    description='Genre:',
    style={'description_width': 'initial'}
)

# Create output area
output = widgets.Output()

# Function to recommend movies
def recommend_by_genre(selected_genre):

    with output:
        clear_output()

        recommendations = movies[
            movies['genres'].str.contains(
                selected_genre, na=False, regex=False
            )
        ]

        recommendations = recommendations.sort_values(
            by='composite_score',
            ascending=False
        )

        print("Recommendations for:", selected_genre)

        display(recommendations[
            ['title', 'genres', 'average_rating',
             'rating_count', 'composite_score']
        ].head(10))

# Connect dropdown
def on_genre_change(change):
    if change['name'] == 'value':
        recommend_by_genre(change['new'])

genre_dropdown.observe(on_genre_change)

# Display
display(genre_dropdown, output)

# Show initial recommendations
recommend_by_genre(genre_dropdown.value)

Dropdown(description='Genre:', options=('Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Do…

Output()

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Movie search box
movie_search = widgets.Text(
    placeholder='Enter movie name...',
    description='Movie:',
    style={'description_width': 'initial'}
)

# Genre dropdown
genre_options = ['All Genres'] + sorted(all_genres)

genre_filter = widgets.Dropdown(
    options=genre_options,
    value='All Genres',
    description='Genre:',
    style={'description_width': 'initial'}
)

# Search button
search_button = widgets.Button(
    description='Search Movies',
    button_style='success',
    icon='search'
)

# Output area
search_output = widgets.Output()

# Search function
def search_movies(button):

    with search_output:
        clear_output()

        search_text = movie_search.value.strip()
        selected_genre = genre_filter.value

        results = movies.copy()

        # Filter by movie title
        if search_text:
            results = results[
                results['title'].str.contains(
                    search_text, case=False, na=False, regex=False
                )
            ]

        # Filter by genre
        if selected_genre != 'All Genres':
            results = results[
                results['genres'].str.contains(
                    selected_genre, case=False,
                    na=False, regex=False
                )
            ]

        # Sort results
        results = results.sort_values(
            by='composite_score',
            ascending=False
        )

        print("Total movies found:", len(results))

        if len(results) == 0:
            print("No matching movies found.")

        else:
            display(results[
                ['title', 'genres', 'average_rating',
                 'rating_count', 'composite_score']
            ].head(20))

# Button click event
search_button.on_click(search_movies)

# Display interface
display(movie_search, genre_filter, search_button, search_output)

Text(value='', description='Movie:', placeholder='Enter movie name...', style=DescriptionStyle(description_wid…

Dropdown(description='Genre:', options=('All Genres', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy'…

Button(button_style='success', description='Search Movies', icon='search', style=ButtonStyle())

Output()

In [12]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Minimum rating count slider
min_votes = widgets.IntSlider(
    value=500,
    min=0,
    max=5000,
    step=100,
    description='Minimum Votes:',
    style={'description_width': 'initial'}
)

# Button
top_button = widgets.Button(
    description='Show Top Movies',
    button_style='success'
)

# Output area
top_output = widgets.Output()

# Function to display top rated movies
def show_top_movies(button):

    with top_output:
        clear_output()

        # Filter movies
        top_movies = movies[
            movies['rating_count'] >= min_votes.value
        ].copy()

        # Sort by average rating
        top_movies = top_movies.sort_values(
            by=['average_rating', 'rating_count'],
            ascending=[False, False]
        )

        print("Top Rated Movies")
        print("Minimum Votes:", min_votes.value)
        print("Movies Found:", len(top_movies))

        display(top_movies[
            ['title', 'genres', 'average_rating',
             'rating_count']
        ].head(10))

# Button click
top_button.on_click(show_top_movies)

# Display interface
display(min_votes, top_button, top_output)

IntSlider(value=500, description='Minimum Votes:', max=5000, step=100, style=SliderStyle(description_width='in…

Button(button_style='success', description='Show Top Movies', style=ButtonStyle())

Output()